<a href="https://colab.research.google.com/github/anyuanay/INFO212/blob/main/INFO212_Week9_Lecture1_Dealing_with_Missing_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

## Week 9: Lecture 1: Dealing with Missing Data (accompanying the lecture-9-1.pptx slides)
---

**Agenda**
- Missing values are None, numpy.nan
- DataFrame.isnull() detects missing values
- DataFrame.notnull() detects non-missing values
- DataFrame.dropna(axis=0, how='any', thresh=None, subset=None, inplace=False)[source]
- DataFrame.fillna(value=None, method=None, axis=None, inplace=False, limit=None, downcast=None)
- DataFrame.transform() performs operations element-wise
- duplicated() returns a boolean Series indicating whether each row is a duplicate
- drop_duplicates() returns a DataFrame where the duplicated array is removed:
- data.replace([-999, -1000], np.nan): replace values

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Messy Data

In [2]:
from google.colab import files
files.upload()

Saving daily-minimum-temperatures-in-me.csv to daily-minimum-temperatures-in-me.csv


{'daily-minimum-temperatures-in-me.csv': b'Date,Daily minimum temperatures\r\n1/1/1981,20.7\r\n1/2/1981,17.9\r\n1/3/1981,18.8\r\n1/4/1981,14.6\r\n1/5/1981,15.8\r\n1/6/1981,15.8\r\n1/7/1981,15.8\r\n1/8/1981,17.4\r\n1/9/1981,21.8\r\n1/10/1981,20\r\n1/11/1981,16.2\r\n1/12/1981,13.3\r\n1/13/1981,16.7\r\n1/14/1981,21.5\r\n1/15/1981,25\r\n1/16/1981,20.7\r\n1/17/1981,20.6\r\n1/18/1981,24.8\r\n1/19/1981,17.7\r\n1/20/1981,15.5\r\n1/21/1981,18.2\r\n1/22/1981,12.1\r\n1/23/1981,14.4\r\n1/24/1981,16\r\n1/25/1981,16.5\r\n1/26/1981,18.7\r\n1/27/1981,19.4\r\n1/28/1981,17.2\r\n1/29/1981,15.5\r\n1/30/1981,15.1\r\n1/31/1981,15.4\r\n2/1/1981,15.3\r\n2/2/1981,18.8\r\n2/3/1981,21.9\r\n2/4/1981,19.9\r\n2/5/1981,16.6\r\n2/6/1981,16.8\r\n2/7/1981,14.6\r\n2/8/1981,17.1\r\n2/9/1981,25\r\n2/10/1981,15\r\n2/11/1981,13.7\r\n2/12/1981,13.9\r\n2/13/1981,18.3\r\n2/14/1981,22\r\n2/15/1981,22.1\r\n2/16/1981,21.2\r\n2/17/1981,18.4\r\n2/18/1981,16.6\r\n2/19/1981,16.1\r\n2/20/1981,15.7\r\n2/21/1981,16.6\r\n2/22/1981,16.5\r

In [3]:
df = pd.read_csv('daily-minimum-temperatures-in-me.csv', parse_dates=['Date'])
df.shape

(3650, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 2 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date                        3650 non-null   datetime64[ns]
 1   Daily minimum temperatures  3650 non-null   object        
dtypes: datetime64[ns](1), object(1)
memory usage: 57.2+ KB


In [5]:
df = df.rename(columns={'Daily minimum temperatures': 'temperatures'})
df.head()

,Date,temperatures
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [6]:
df.plot(x='Date', y='temperatures')

TypeError: no numeric data to plot

In [7]:
df[df['temperatures'].apply(lambda x: not str(x).replace('.', '', 1).isdigit())]

,Date,temperatures
565,1982-07-20,?0.2
566,1982-07-21,?0.8
1290,1984-07-14,?0.1


In [8]:
# list the rows that temperature can't be evaluated as numeric
df[pd.to_numeric(df['temperatures'], errors='coerce').isnull()]

,Date,temperatures
565,1982-07-20,?0.2
566,1982-07-21,?0.8
1290,1984-07-14,?0.1


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          3650 non-null   datetime64[ns]
 1   temperatures  3650 non-null   object        
dtypes: datetime64[ns](1), object(1)
memory usage: 57.2+ KB


# Handling Missing Data

Missing data occurs commonly in many data analysis applications.
All of the descriptive statistics on pandas objects exclude missing data by default. For numeric data, pandas uses the floating-point value NaN (Not a Number) to represent missing data. We call this a sentinel value that can be easily detected.

## `isnull()`: detects missing values
```
string_data = pd.Series(['aardvark', 'artichoke', np.nan, 'avocado'])

string_data.isnull()

string_data.isnull().sum()
```

## Reasons for Missing Values

Before we start treating the missing values, it is important to understand the various reasons for the missingness in data. Broadly speaking, there can be three possible reasons: (references: Wikipedia: https://en.wikipedia.org/wiki/Missing_data ; Flexible Imputation of Missing Data by R: https://stefvanbuuren.name/fimd/sec-MCAR.html)

1. Missing Completely at Random (MCAR): The missing values on a given variable (Y) are not associated with other variables in a given data set or with the variable (Y) itself. In other words, there is no particular reason for the missing values. An example of MCAR is a weighing scale that ran out of batteries. Some of the data will be missing simply because of bad luck.

2. Missing at Random (MAR): MAR occurs when the missingness is not random, but where missingness can be fully accounted for by variables where there is complete information. For example, when placed on a soft surface, a weighing scale may produce more missing values than when placed on a hard surface. Such data are thus not MCAR. If, however, we know surface type and if we can assume MCAR within the type of surface, then the data are MAR.

3. Missing Not at Random (MNAR): Missingness depends on unobserved data or the value of the missing data itself. MNAR means that the probability of being missing varies for reasons that are unknown to us. For example, the weighing scale mechanism may wear out over time, producing more missing data as time progresses, but we may fail to note this.

## Dealing with Missing Data
Missing data are often denoted by a "na, n/a, N/a, N/A (not applicable)" or "Nan, NaN, NAN (not a number)". The denomination depends on the used progamming langauge (NaN is the most common in python language for machine learning). In general, there are two approaches to handle missing data: Deletion and Imputation.

## Deletions

![](https://imgur.com/tBvdfyX.png)

Deletion means to delete the missing values from a dataset. Deletions are further categorized into three types:

### Pairwise Deletion

>Parwise Deletion is used when values are missing completely at random i.e MCAR. During Pairwise deletion, only the missing values are deleted. All operations in pandas like mean,sum etc intrinsically skip missing values.

### Listwise Deletion/ Dropping rows

>During Listwise deletion, complete rows (which contain the missing values) are deleted. As a result, it is also called Complete Case deletion. Like Pairwise deletion, listwise deletions are also only used for MCAR values.

### Dropping complete columns

>If a column contains a lot of missing values, say more than 80%, and the feature is not significant, you might want to delete that feature. However, again, it is not a good methodology to delete data.

## Imputations

![](https://imgur.com/bL0iHde.png)

>Imputation refers to replacing missing data with substituted values. There are a lot of ways in which the missing values can be imputed depending upon the nature of the problem and data. Dependng upon the nature of the problem, imputation techniques can be broadly they can be classified as follows:


### Basic Imputation Techniques
  
  - Imputating with a constant value
  - Imputation using the statistics (mean, median or most frequent) of each column in which the missing values are located

For this we shall use the `The SimpleImputer` class from sklearn.

## Filling missing data
We can fill in the missing value using the previous or next value by `ffill` or `bfill`.
```
df = pd.DataFrame(np.random.randn(6, 3))
df.iloc[2:, 1] = NA
df.iloc[4:, 2] = NA

data.fillna(method='bfill')

df.fillna(method='ffill', limit=2)
```

# Data Transformation

## The transform() function:
- The transform() function in pandas is used to perform operations on each element of a series or column in a DataFrame.
- Unlike apply(), transform() returns the results in the same row order and same length as the original DataFrame.
- This makes it particularly useful for filling missing values.
- The transform() function is often used with groupby objects to perform operations within groups while maintaining the original DataFrame shape.
```
df = pd.DataFrame({'A': ['foo', 'bar', 'foo', 'bar', 'foo', 'bar', ],
                 'B': [1, np.nan, 2, 4, np.nan, 2]})
B_imputed = df.groupby('A')['B'].transform(lambda x: x.fillna(x.mean()))
```

## Exercise:
Run and test the above code.

# Replacing Values
Filling in missing data with the fillna method is a special case of more general value
replacement. As you’ve already seen, map can be used to modify a subset of values in
an object.
## `replace` provides a simpler and more flexible way to do so. Let’s consider this Series:

```
data = pd.Series([1., -999., 2., -999., -1000., 3.])
```

The -999 values might be sentinel values for missing data. To replace these with NA
values that pandas understands, we can use replace, producing a new Series (unless
you pass inplace=True):

```
data1 = data.replace([-999, -1000], np.nan)

data1.fillna(data1.mean())
```

## Exercise:
Run and test the above code.

If you want to replace multiple values at once, you instead pass a list and then the
substitute value:

```
data.replace([-999, -1000], np.nan)
```

To use a different replacement for each value, pass a list of substitutes:

```
data.replace([-999, -1000], [np.nan, 0])
```

The argument passed can also be a dict:

```
data.replace({-999: np.nan, -1000: 0})
```

# Retrieval Practice